# Phase 1.2 — Dataset Inventory

This notebook inventories the raw Retailrocket dataset without modifying raw files or creating processed data.

It is designed for the large CSV files and uses chunked reads where full-file loading would be unnecessary.

In [16]:
from pathlib import Path
import csv
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

print('Project root :', PROJECT_ROOT.resolve())
print('Raw data dir:', RAW_DIR.resolve())
            

Project root : F:\annuspeaks.com\recommendation-system
Raw data dir: F:\annuspeaks.com\recommendation-system\data\raw


## Inventory Targets

Raw files:

- `events.csv` — behavioral interactions
- `item_properties_part1.csv` — product properties
- `item_properties_part2.csv` — product properties
- `category_tree.csv` — category hierarchy

**Important:** Retailrocket does not expose separate raw columns named price, brand, or description. Those concepts must be investigated through the generic `property` / `value` structure.

In [17]:
files = [
    'events.csv',
    'item_properties_part1.csv',
    'item_properties_part2.csv',
    'category_tree.csv',
]

inventory = []
for name in files:
    p = RAW_DIR / name
    inventory.append({
        'file': name,
        'exists': p.exists(),
        'size_mb': round(p.stat().st_size / (1024**2), 2) if p.exists() else None,
    })

display(pd.DataFrame(inventory))

,file,exists,size_mb
0,events.csv,True,89.87
1,item_properties_part1.csv,True,461.88
2,item_properties_part2.csv,True,389.99
3,category_tree.csv,True,0.01


In [18]:
def read_header(path):
    with path.open('r', encoding='utf-8', errors='replace', newline='') as f:
        return next(csv.reader(f))

schemas = []
for name in files:
    p = RAW_DIR / name
    if p.exists():
        schemas.append({'file': name, 'columns': read_header(p)})

for row in schemas:
    print(f"\n{row['file']}")
    for col in row['columns']:
        print('  -', col)


events.csv
  - timestamp
  - visitorid
  - event
  - itemid
  - transactionid

item_properties_part1.csv
  - timestamp
  - itemid
  - property
  - value

item_properties_part2.csv
  - timestamp
  - itemid
  - property
  - value

category_tree.csv
  - categoryid
  - parentid


In [19]:
# Product-property inventory across both large property files.
# This intentionally inventories property IDs and representative values without
# building a potentially huge per-property set of item IDs in RAM.
property_files = ['item_properties_part1.csv', 'item_properties_part2.csv']
property_counts = {}
property_examples = {}
property_min_ts = {}
property_max_ts = {}

for name in property_files:
    path = RAW_DIR / name
    for chunk in pd.read_csv(path, chunksize=250_000):
        for prop, group in chunk.groupby('property', dropna=False):
            key = str(prop)
            property_counts[key] = property_counts.get(key, 0) + len(group)
            property_examples.setdefault(key, [])
            if len(property_examples[key]) < 5:
                remaining = 5 - len(property_examples[key])
                property_examples[key].extend(
                    group['value'].dropna().astype(str).head(remaining).tolist()
                )
            ts = pd.to_numeric(group['timestamp'], errors='coerce')
            if ts.notna().any():
                cmin, cmax = ts.min(), ts.max()
                property_min_ts[key] = cmin if key not in property_min_ts else min(property_min_ts[key], cmin)
                property_max_ts[key] = cmax if key not in property_max_ts else max(property_max_ts[key], cmax)

property_inventory = pd.DataFrame([
    {
        'property': k,
        'rows': property_counts[k],
        'example_values': ' | '.join(property_examples[k]),
        'min_timestamp': property_min_ts.get(k),
        'max_timestamp': property_max_ts.get(k),
    }
    for k in property_counts
]).sort_values(['rows', 'property'], ascending=[False, True])

print('Unique property IDs:', len(property_inventory))
display(property_inventory)


Unique property IDs: 1104


,property,rows,example_values,min_timestamp,max_timestamp
822,888,3000398,1116713 960601 n277.200 | 1038400 45956 n504.0...,1431226800000,1442113200000
727,790,1790516,n15360.000 | n21000.000 | n5400.000 | n39588.0...,1431226800000,1442113200000
927,available,1503639,0 | 0 | 0 | 1 | 0,1431226800000,1442113200000
928,categoryid,788214,1338 | 1277 | 1059 | 1147 | 47,1431226800000,1442113200000
546,6,631471,319724 | 1214748 1186610 | 377560 | 153715 484...,1431226800000,1442113200000
...,...,...,...,...,...
1099,634,1,168817,1439089200000,1439089200000
1090,722,1,769062,1439694000000,1439694000000
1089,744,1,1014338,1439089200000,1439089200000
1103,769,1,1217479,1435460400000,1435460400000


In [20]:
# Category-tree inventory
category_path = RAW_DIR / 'category_tree.csv'
categories = pd.read_csv(category_path)
print('Rows:', f'{len(categories):,}')
print('Unique categoryid:', f"{categories['categoryid'].nunique(dropna=True):,}")
print('Unique parentid  :', f"{categories['parentid'].nunique(dropna=True):,}")
print('Root categories (parentid is null):', f"{categories['parentid'].isna().sum():,}")
            

Rows: 1,669
Unique categoryid: 1,669
Unique parentid  : 362
Root categories (parentid is null): 25


In [21]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

events_path = RAW_DIR / "events.csv"
property_files = [
    "item_properties_part1.csv",
    "item_properties_part2.csv"
]

event_items = set()

for chunk in pd.read_csv(
    events_path,
    usecols=["itemid"],
    chunksize=250_000
):
    event_items.update(
        chunk["itemid"].dropna().astype(str).unique()
    )

property_items = set()

for name in property_files:
    path = RAW_DIR / name

    for chunk in pd.read_csv(
        path,
        usecols=["itemid"],
        chunksize=250_000
    ):
        property_items.update(
            chunk["itemid"].dropna().astype(str).unique()
        )

print("Unique items in events        :", f"{len(event_items):,}")
print("Unique items in item props    :", f"{len(property_items):,}")
print("Event items with properties   :", f"{len(event_items & property_items):,}")
print("Event items without properties:", f"{len(event_items - property_items):,}")

Unique items in events        : 235,061
Unique items in item props    : 417,053
Event items with properties   : 185,246
Event items without properties: 49,815


In [22]:
# Product -> category relationship check
# Retailrocket's raw files do not contain a direct itemid-categoryid table.
# We therefore check whether any property/value records expose category-like
# semantics and separately inspect the category hierarchy itself.

category_path = RAW_DIR / 'category_tree.csv'
categories = pd.read_csv(category_path)

print('CATEGORY TREE')
print('Rows:', f'{len(categories):,}')
print('Unique categories:', f"{categories['categoryid'].nunique(dropna=True):,}")
print('Root categories:', f"{categories['parentid'].isna().sum():,}")

# Search a bounded sample of property values for category-like fields/values.
candidate_props = []
for name in ['item_properties_part1.csv', 'item_properties_part2.csv']:
    path = RAW_DIR / name
    sample = pd.read_csv(path, nrows=500_000, usecols=['itemid', 'property', 'value'])
    mask = sample['property'].astype(str).str.lower().str.contains(
        'category|cat', regex=True, na=False
    )
    if mask.any():
        candidate_props.append(sample.loc[mask].copy())

if candidate_props:
    category_like = pd.concat(candidate_props, ignore_index=True)
    print('\nCategory-like property records found in audit samples:')
    display(category_like.head(20))
    print('Candidate property IDs:', category_like['property'].astype(str).unique().tolist())
else:
    print('\nNo category-like property ID was identified in the sampled property records.')

print('\nConclusion: category_tree.csv provides the category hierarchy, but a direct')
print('itemid -> categoryid mapping must be established from the raw property semantics')
print('if such a mapping exists; it must not be assumed from the hierarchy alone.')


CATEGORY TREE
Rows: 1,669
Unique categories: 1,669
Root categories: 25

Category-like property records found in audit samples:


,itemid,property,value
0,460429,categoryid,1338
1,281245,categoryid,1277
2,35575,categoryid,1059
3,8313,categoryid,1147
4,55102,categoryid,47
5,397079,categoryid,619
6,265036,categoryid,1228
7,124459,categoryid,1277
8,350508,categoryid,546
9,221365,categoryid,1226


Candidate property IDs: ['categoryid']

Conclusion: category_tree.csv provides the category hierarchy, but a direct
itemid -> categoryid mapping must be established from the raw property semantics
if such a mapping exists; it must not be assumed from the hierarchy alone.


## Phase 1.2 Review Checklist

After running all cells, document:

- Exact file sizes and row counts.
- Exact schemas and key fields.
- Full event distribution and identifier cardinalities.
- Timestamp coverage.
- Full property-ID inventory and representative values.
- Category hierarchy size and root structure.
- Cross-file item-ID coverage.
- Which product-property IDs plausibly encode category, price, brand, description, or other attributes.

**Do not clean, transform, or move raw data during this phase.**

In [23]:
# Product -> Category relationship check

print("events.csv columns:", list(pd.read_csv(
    RAW_DIR / "events.csv", nrows=1
).columns))

print("\ncategory_tree.csv columns:", list(
    pd.read_csv(RAW_DIR / "category_tree.csv", nrows=1).columns
))

# Search property IDs for category-like semantics
for name in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    sample = pd.read_csv(
        RAW_DIR / name,
        nrows=500_000,
        usecols=["itemid", "property", "value"]
    )

    mask = sample["property"].astype(str).str.lower().str.contains(
        "category|cat", regex=True, na=False
    )

    print(f"\n{name} — category-like records:")
    display(sample.loc[mask].head(20))

events.csv columns: ['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

category_tree.csv columns: ['categoryid', 'parentid']

item_properties_part1.csv — category-like records:


,itemid,property,value
0,460429,categoryid,1338
140,281245,categoryid,1277
151,35575,categoryid,1059
189,8313,categoryid,1147
197,55102,categoryid,47
213,397079,categoryid,619
237,265036,categoryid,1228
254,124459,categoryid,1277
310,350508,categoryid,546
325,221365,categoryid,1226



item_properties_part2.csv — category-like records:


,itemid,property,value
15,8921,categoryid,1188
70,122405,categoryid,769
162,225336,categoryid,491
182,193256,categoryid,1261
257,301841,categoryid,1493
273,187011,categoryid,928
293,247243,categoryid,1167
299,364449,categoryid,610
319,312760,categoryid,1059
326,184587,categoryid,1277
